# Phase B v2 — GPU dataset generation (sim-v2.0)

Generates the **7-class / 9-band enhanced-Motley-Keenan** training set on a GPU.
This is the fast path: the physics engine is ported to torch and runs on CUDA,
so 2,500 positions × 9 bands takes minutes instead of hours.

**Runtime → GPU** (Runtime ▸ Change runtime type ▸ GPU; an A100 or L4 is ideal,
a T4 is fine).

What this notebook does, in order:
1. clone the repo and install deps
2. **build the 7-class v2 grid** from the deployed v1 grid (the interior-glass
   un-fold) and write `manifest_v2.json`
3. **check numpy↔torch parity** — the GPU engine must match the CPU reference
4. **generate** the dataset on GPU, resumably, to Google Drive
5. **audit** the result (split disjointness, clip fractions per band)

Nothing here trains a model — that is `phase_c_v2_train.ipynb`. This notebook
only produces `dataset_v2/`.

## 0. Repo + dependencies

The engine modules (`physics_v2.py`, `engine_v2.py`, `engine_v2_torch.py`) live
in `SIM/`; the v2 driver scripts in `SIM V2/`. Torch and scipy are the only
requirements and both are preinstalled on Colab.

In [ ]:
import os, subprocess, sys
REPO = "indoor-walk-test"
if not os.path.exists(REPO):
    r = subprocess.run(["git","clone","--depth","1",
        "https://github.com/cgm2179/indoor-walk-test.git"], capture_output=True, text=True)
    if r.returncode != 0:  # private repo -> ask for a token
        import getpass
        tok = getpass.getpass("GitHub token (Contents: Read): ")
        subprocess.run(["git","clone","--depth","1",
            f"https://{tok}@github.com/cgm2179/indoor-walk-test.git"], check=True)
%cd indoor-walk-test
import torch
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")
assert torch.cuda.is_available(), "Set Runtime -> GPU before running"


## 1. Mount Drive (dataset lands here, so generation is resumable)

If Colab times out mid-run, re-running the generate cell picks up where it
stopped: finished shards are skipped.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_OUT = "/content/drive/MyDrive/SIM/dataset_v2"     # <- shards go here
os.makedirs(DRIVE_OUT, exist_ok=True)
print("dataset ->", DRIVE_OUT)


## 2. Build the 7-class v2 grid (the interior-glass un-fold)

v1 folded the lunch-room / meeting-room glass into the single exterior glass
class because both were ~3 dB. v2 measures the FCC facade as **20 dB low-E**
glass, so the two can no longer share a class — the grid becomes **7-class**
and the surrogate input grows to `IN_CH=10`.

This reads the deployed v1 grid (`SIM/grid_model.npy`) and the interior-glass
regions from `STEP_1/material_overrides.json`, writes `SIM V2/grid_model_v2.npy`
and `SIM V2/manifest_v2.json`.

> If `material_overrides.json` has no interior-glass (id 6) entry yet, class 6
> comes out empty and the grid is 7-class-capable but effectively still 6-class.
> Add the lunch-room enclosure as an id-6 override first (MODEL_CARD_V2 §5.2).

In [ ]:
import subprocess
r = subprocess.run([sys.executable, "SIM V2/phase_a_v2.py", "--prepare",
                    "--repo", ".", "--out", "SIM V2"], capture_output=True, text=True)
print(r.stdout); print(r.stderr)


## 3. Parity check — the GPU engine must match the CPU reference

`engine_v2.py` (numpy) is the source of truth. This asserts the torch backend
agrees with it: obstruction near bit-identical, path loss p99.99 < 0.05 dB.
If this fails, **stop** — do not generate a dataset from an unverified engine.

In [ ]:
r = subprocess.run([sys.executable, "SIM/engine_v2_torch.py", "--parity",
                    "--device", "cuda"], capture_output=True, text=True)
print(r.stdout); print(r.stderr)
assert "PARITY OK" in r.stdout, "GPU/CPU parity failed — inspect before generating"


## 4. Throughput estimate

A quick benchmark so you know how long the full run will take on this GPU.

In [ ]:
r = subprocess.run([sys.executable, "SIM/engine_v2_torch.py", "--bench",
                    "--device", "cuda"], capture_output=True, text=True)
print(r.stdout); print(r.stderr)


## 5. Smoke test (12 positions) before committing to the full run

Confirms the whole path — grid, engine, sharding, Drive write — works before
you spend GPU quota on 2,500 positions.

In [ ]:
r = subprocess.run([sys.executable, "SIM V2/phase_b_v2_generate.py",
    "--smoke", "--device", "cuda",
    "--grid", "SIM V2/grid_model_v2.npy",
    "--inside", "SIM V2/inside_mask_v2.npy",
    "--walkable", "SIM V2/walkable_mask_v2.npy",
    "--manifest", "SIM V2/manifest_v2.json",
    "--out", "/content/smoke_v2"], capture_output=True, text=True)
print(r.stdout[-2000:]); print(r.stderr[-2000:])


## 6. Generate the full dataset (2,500 positions × 9 bands → Drive)

Resumable: re-run this cell after any timeout and it continues. For a multi-GPU
sweep, run several copies with `--shard-mod N --shard-rem 0..N-1`.

Expect a few GB of float16 targets. The per-band clip fractions are printed by
the audit in the next cell; the high bands (5.5 / 6 GHz) clip more than v1
because the frequency-scaled losses are genuinely larger — the wider v2 clip
window already accounts for this.

In [ ]:
import time
t0 = time.time()
proc = subprocess.Popen([sys.executable, "SIM V2/phase_b_v2_generate.py",
    "--device", "cuda",
    "--grid", "SIM V2/grid_model_v2.npy",
    "--inside", "SIM V2/inside_mask_v2.npy",
    "--walkable", "SIM V2/walkable_mask_v2.npy",
    "--manifest", "SIM V2/manifest_v2.json",
    "--out", DRIVE_OUT],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:      # stream progress live
    print(line, end="")
proc.wait()
print(f"\n[generate finished in {(time.time()-t0)/60:.1f} min, rc={proc.returncode}]")


## 7. Audit the dataset

Split disjointness, all-freqs-together per position, target histogram, and the
per-band clip fractions. This is the gate before training.

In [ ]:
r = subprocess.run([sys.executable, "SIM V2/phase_b_v2_generate.py", "--audit",
    "--manifest", "SIM V2/manifest_v2.json", "--out", DRIVE_OUT],
    capture_output=True, text=True)
print(r.stdout); print(r.stderr)


## 8. Next step

The dataset is in ``DRIVE_OUT``. Open **`phase_c_v2_train.ipynb`** (fresh from GitHub,
GPU runtime) to train the `IN_CH=10`, 7-class UNet on it. That notebook points
its dataset loader at this Drive folder.

Every material constant in this dataset is literature (ITU-R P.2040 + the fitted
sub-resolution excess), not yet calibrated to this building. Once a known-Tx
walk exists, Phase D fits the per-material scales and you regenerate + fine-tune
— which is exactly what the `jitter` field in each shard makes cheap.